# Financial Reasoning — Qwen3-8B Fine-tuning (Fino1 Approach)

**Based on:** [Fino1: On the Transferability of Reasoning-Enhanced LLMs to Finance](https://arxiv.org/abs/2502.08127)  
**Model:** `Qwen3-8B` (native thinking mode) → [TheFinAI/Fin-o1-8B](https://huggingface.co/TheFinAI/Fin-o1-8B) is the reference  
**Data:** Same `TheFinAI/FinCoT` dataset as Project 1 — no reprocessing needed

## Why switch from Project 1 (Qwen2.5-7B)?

| | Project 1 (Qwen2.5-7B) | This notebook (Qwen3-8B Fino1) |
|---|---|---|
| Baseline accuracy | 31.4% | **72%** |
| SFT data format | Raw text (manual `<\|im_start\|>` tokens) | **Conversational messages dict** |
| EOS token | Manual workaround needed | Auto-handled by Qwen3 tokenizer |
| Learning rate | 5e-6 (too low, loss never converged) | **2e-5** (Fino1 value) |
| max_seq_length | 2048 (OOM fix — caused truncation) | **4096** (4-bit loads to handle memory) |
| After SFT gain | +0.9pp (format rates *decreased*) | Target: **+5pp** → ≥77% |

## Run order

| Section | What | Time |
|---|---|---|
| 0 · Setup | GPU, install, auth, upload data, clone repo | ~10 min |
| 1 · Baseline | Optional — Qwen3-8B already evaluated at 72% | skip |
| 2 · SFT | Fino1-style SFT on FinCoT | ~4–6 h |
| 3 · GRPO | RL to push accuracy further | ~6–10 h |
| 4 · Eval & Push | Compare vs baseline, push to HF | ~20 min |

---
## 0-A · Runtime Check

> ⚠️ **Always use A100** for this notebook. Qwen3-8B with 4096 context needs ~20 GB VRAM.  
> Runtime → Change runtime type → GPU → A100

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Go to Runtime → Change runtime type → GPU (A100)')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU : {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')

# Unlike Project 1, we always use Qwen3-8B — no model size switching based on VRAM.
# 4-bit quantization keeps VRAM usage ~18-20 GB even on T4, but A100 is recommended
# for comfortable training at seq_len=4096.
BASE_MODEL = 'unsloth/Qwen3-8B'
print(f'\nModel : {BASE_MODEL}')
print(f'Config: config/sft_qwen3_v1.yaml')

---
## 0-B · Install Dependencies

> Same packages as Project 1. Unsloth is installed first to ensure it patches TRL/transformers correctly.

In [ ]:
# Unsloth must be installed before everything else
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' -q
!pip install trl>=0.12.0 transformers>=4.45.0 datasets>=2.20.0 peft>=0.12.0 -q
!pip install wandb openai pyyaml python-dotenv huggingface_hub tqdm -q
print('\n✅ All packages installed')

---
## 0-C · Upload Data

**Same FinCoT files as Project 1 — no reprocessing needed.**

The Fino1 paper uses the full `TheFinAI/FinCoT` dataset with difficulty-aware filtering.  
Our `01_data_pipeline.py` already applied difficulty + quality filters, which aligns with this philosophy.  
The 5,841 quality-filtered examples are a *stricter* subset — better signal for SFT.

Upload the same 3 files from your Mac's `data/processed/` to Google Drive:
- `sft_train_v1.jsonl`
- `sft_val_v1.jsonl`  
- `rl_raw.jsonl`

In [ ]:
from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/llm-ft/processed'
LOCAL_DATA = '/content/project/data/processed'

Path(LOCAL_DATA).mkdir(parents=True, exist_ok=True)

for fname in ['sft_train_v1.jsonl', 'sft_val_v1.jsonl', 'rl_raw.jsonl']:
    src = f'{DRIVE_DATA}/{fname}'
    dst = f'{LOCAL_DATA}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'✅ {fname} ({size_mb:.0f} MB)')
    else:
        print(f'❌ Not found: {src}')

In [ ]:
# Validate data + preview the format that will be used for training
import json, re
from pathlib import Path

LOCAL_DATA = '/content/project/data/processed'

for fname in ['sft_train_v1.jsonl', 'sft_val_v1.jsonl', 'rl_raw.jsonl']:
    path = Path(LOCAL_DATA) / fname
    if not path.exists():
        print(f'❌ Missing: {fname}')
        continue
    examples = [json.loads(l) for l in open(path)]
    has_think = sum(1 for ex in examples
                    if re.search(r'<think>.*?</think>', ex.get('reasoning_trace',''), re.DOTALL))
    print(f'✅ {fname}: {len(examples)} examples | think tag: {has_think/len(examples):.0%}')

# Show what the Fino1 conversational format will look like
print('\n--- Example training format (Fino1 conversational) ---')
ex = json.loads(open(f'{LOCAL_DATA}/sft_train_v1.jsonl').readline())
trace = ex['reasoning_trace']
think = re.search(r'<think>(.*?)</think>', trace, re.DOTALL)
answer = re.search(r'<answer>(.*?)</answer>', trace, re.DOTALL)

print('messages[')
print('  {role: system, content: "You are a financial analyst expert..."},')
print(f'  {{role: user, content: "Financial Context:\\n{ex["context"][:80]}...\\n\\nQuestion: {ex["question"][:60]}..."}}')
if think and answer:
    print(f'  {{role: assistant, content: "<think>\\n{think.group(1)[:80]}...\\n</think>\\n<answer>{answer.group(1)}</answer>"}}')
print(']')

---
## 0-D · Authentication

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY']      = userdata.get('WANDB_API_KEY')
    os.environ['HF_TOKEN']           = userdata.get('HF_TOKEN')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    print('✅ Loaded secrets from Colab Secrets')
except Exception:
    os.environ['WANDB_API_KEY']      = 'paste_here'
    os.environ['HF_TOKEN']           = 'paste_here'
    os.environ['OPENROUTER_API_KEY'] = 'paste_here'
    print('⚠️  Using hardcoded keys — use Colab Secrets instead')

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('✅ W&B and HuggingFace authenticated')

---
## 0-E · Clone Project Repo

In [ ]:
import os
from pathlib import Path

REPO_URL    = 'https://github.com/tiffanychum/llm-fine-tunning-financial-reasoning.git'
PROJECT_DIR = '/content/project'

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    # Force-sync with remote (discards any local config changes)
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} reset --hard origin/master

%cd {PROJECT_DIR}

Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('results').mkdir(exist_ok=True)
Path('checkpoints').mkdir(exist_ok=True)

print('Working dir:', os.getcwd())
!ls scripts/
!ls config/

---
## 1 · Baseline (Reference Only)

Qwen3-8B was already evaluated on the val set (50-sample fast check).  
**No need to rerun** unless you want the full 650-example baseline.

| Metric | Qwen3-8B (no tuning) | Qwen2.5-7B (no tuning) |
|---|---|---|
| Accuracy | **72%** | 31.4% |
| `<think>` tag | 4% | 94% |
| `<answer>` tag | 94% | 94% |

Qwen3-8B starts from a much stronger baseline — SFT goal is to add consistent
`<think>` tag usage and improve accuracy, not teach reasoning from scratch.

In [ ]:
# Reference baseline — already known from local evaluation
BASELINE_ACCURACY = 0.72
SFT_TARGET        = BASELINE_ACCURACY + 0.05  # target +5pp → 77%

print(f'Qwen3-8B baseline : {BASELINE_ACCURACY:.1%}')
print(f'SFT target        : {SFT_TARGET:.1%} (+5pp)')
print(f'GRPO target       : {SFT_TARGET + 0.03:.1%} (+3pp over SFT)')
print(f'\nComparison: Fin-R1 (same algo, 60k examples) achieved 75.2%')
print(f'Our target with 5,841 quality examples: ≥77% (same model family)')

---
## 2 · SFT Training (Fino1 / Qwen3-8B)

**What's different from Project 1:**

| | Project 1 `02_sft_train.py` | This `02_sft_qwen3.py` |
|---|---|---|
| Data format | Raw text string (manual chat tokens) | **Conversational messages dict** |
| Chat template | Hand-built `<\|im_start\|>` strings | **TRL auto-applies Qwen3 template** |
| EOS token | `tokenizer.eos_token = "<\|im_end\|>"` workaround | Not needed — Qwen3 sets it correctly |
| LR | 5e-6 (too low, loss never converged) | **2e-5** (Fino1 model card) |
| Seq length | 2048 (OOM fix — truncated `<answer>` tags) | **4096** with 4-bit loading |
| Packing | True (caused format degradation) | **False** (per-example sequences) |
| System prompt | Generic financial assistant | **Fino1 system prompt** (exactly matches paper) |

In [ ]:
# Verify config before training
import yaml

with open('config/sft_qwen3_v1.yaml') as f:
    cfg = yaml.safe_load(f)

print('=== Fino1 / Qwen3-8B SFT Config ===')
print(f'  model        : {cfg["model"]["name"]}')
print(f'  max_seq_len  : {cfg["model"]["max_seq_length"]}  (restored from 2048)')
print(f'  load_in_4bit : {cfg["model"]["load_in_4bit"]}   (4-bit to fit 4096 context)')
print(f'  lora r       : {cfg["lora"]["r"]}  (down from 32 — Qwen3 needs less adaptation)')
print(f'  learning_rate: {cfg["training"]["learning_rate"]}  (Fino1 value, 4× higher than failed run)')
print(f'  eff. batch   : {cfg["training"]["per_device_batch_size"] * cfg["training"]["gradient_accumulation_steps"]}  (matches Fino1 16-sample batch)')
print(f'  packing      : {cfg["training"]["packing"]}  (off — prevents format degradation)')
print(f'  epochs       : {cfg["training"]["epochs"]}')

In [ ]:
# Run SFT — Fino1 approach
# ~4-6h on A100 for 5,841 examples × 3 epochs
# Watch W&B for:
#   - train/loss should drop below 1.0 (vs Project 1 which stalled at 1.15)
#   - eval/loss should track train/loss (no large gap = no overfitting)
#   - if loss plateaus before epoch 3, training is working correctly

!python scripts/02_sft_qwen3.py --config config/sft_qwen3_v1.yaml

In [ ]:
# If the session disconnected, resume from latest checkpoint:
# !python scripts/02_sft_qwen3.py --config config/sft_qwen3_v1.yaml --resume

In [ ]:
# Evaluate SFT model on val set
!python scripts/00_setup_eval.py \
    --model checkpoints/sft_qwen3/best \
    --output results/sft_qwen3/baseline_scores.json \
    --batch_size 4

import json

with open('results/sft_qwen3/baseline_scores.json') as f:
    sft_result = json.load(f)

SFT_ACCURACY = sft_result['metrics']['accuracy']

print('=== SFT RESULTS ===')
print(f'Baseline (Qwen3-8B) : {BASELINE_ACCURACY:.1%}')
print(f'After SFT           : {SFT_ACCURACY:.1%}  ({(SFT_ACCURACY-BASELINE_ACCURACY)*100:+.1f} pp)')
print(f'<think> tag rate    : {sft_result["metrics"]["format_think_pct"]:.1%}  (was 4% at baseline)')
print(f'<answer> tag rate   : {sft_result["metrics"]["format_answer_pct"]:.1%}')

# Note: unlike Project 1 where format rates DECREASED after SFT (sign of truncation),
# we expect <think> rate to jump from 4% → 90%+ after Fino1-style SFT.

if SFT_ACCURACY >= BASELINE_ACCURACY + 0.05:
    print('\n✅ Gate passed (+5pp) — proceed to GRPO')
elif SFT_ACCURACY >= BASELINE_ACCURACY + 0.02:
    print('\n⚠️  Modest gain — check W&B loss curves, then proceed to GRPO')
else:
    print('\n❌ Small gain — check loss curves before GRPO')
    print('   Likely cause: LR still off, or format issue in training')

---
## 3 · GRPO Training

**Key Fino1 finding:** GRPO yields reliable gains for financial reasoning.  
PPO and DPO both failed to improve in the paper's systematic comparison.
Your choice of GRPO is empirically validated.

**RL data strategy (aligned with Fin-R1):**  
The `filter_solvable()` function keeps only questions where the SFT model gets ≥1 correct  
out of 8 attempts. This matches Fin-R1's approach of using *hard-but-solvable* examples  
for RL — exactly the right variance for GRPO learning signal.

> ⚠️ Update `config/grpo_v1.yaml` to point `model.sft_checkpoint` at `checkpoints/sft_qwen3/best`

In [ ]:
from pathlib import Path

SFT_CKPT = 'checkpoints/sft_qwen3/best'
assert Path(SFT_CKPT).exists(), f'SFT checkpoint missing at {SFT_CKPT}'
print(f'✅ SFT checkpoint found: {SFT_CKPT}')
!ls {SFT_CKPT}

In [ ]:
# Point GRPO config at the Qwen3 SFT checkpoint
import yaml

with open('config/grpo_v1.yaml') as f:
    grpo_cfg = yaml.safe_load(f)

grpo_cfg['model']['sft_checkpoint'] = 'checkpoints/sft_qwen3/best'
grpo_cfg['experiment']['name']      = 'grpo-qwen3-8b-v1'
grpo_cfg['experiment']['tags']      = ['grpo', 'qwen3-8b', 'fino1', 'financial-reasoning']
grpo_cfg['output']['checkpoint_dir'] = 'checkpoints/grpo_qwen3/'
grpo_cfg['output']['results_dir']    = 'results/grpo_qwen3/'

with open('config/grpo_v1.yaml', 'w') as f:
    yaml.dump(grpo_cfg, f)

print('GRPO config updated:')
print(f'  sft_checkpoint : {grpo_cfg["model"]["sft_checkpoint"]}')
print(f'  experiment     : {grpo_cfg["experiment"]["name"]}')

In [ ]:
# Run GRPO — ~6-10h on A100
# Watch W&B for:
#   - reward/mean should increase over time
#   - entropy should stay > 0.15 (below = collapse signal)
#   - format_rate should stay > 70%

!python scripts/03_grpo_train.py --config config/grpo_v1.yaml

In [ ]:
# Final evaluation — compare all phases
!python scripts/00_setup_eval.py \
    --model checkpoints/grpo_qwen3/best \
    --output results/grpo_qwen3/scores.json \
    --batch_size 4

import json

with open('results/grpo_qwen3/scores.json') as f:
    grpo_result = json.load(f)

GRPO_ACCURACY = grpo_result['metrics']['accuracy']

print('=== FINAL COMPARISON ===')
print(f'Qwen3-8B baseline : {BASELINE_ACCURACY:.1%}')
print(f'After SFT         : {SFT_ACCURACY:.1%}  ({(SFT_ACCURACY-BASELINE_ACCURACY)*100:+.1f} pp)')
print(f'After SFT + GRPO  : {GRPO_ACCURACY:.1%}  ({(GRPO_ACCURACY-BASELINE_ACCURACY)*100:+.1f} pp)')
print(f'\nComparison to Fin-R1 (same algo, 60k examples):')
print(f'  Fin-R1 final    : 75.2%  (+9.6pp over its 65.6% baseline)')
print(f'  Our final       : {GRPO_ACCURACY:.1%}  ({(GRPO_ACCURACY-0.656)*100:+.1f} pp vs Fin-R1 baseline)')

BEST_MODEL = 'checkpoints/grpo_qwen3/best' if GRPO_ACCURACY >= SFT_ACCURACY else 'checkpoints/sft_qwen3/best'
print(f'\nBest model: {BEST_MODEL}')

---
## 4 · Push to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

HF_REPO_ID = 'YOUR_USERNAME/financial-reasoning-qwen3'  # ← change this

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
api.upload_folder(folder_path=BEST_MODEL, repo_id=HF_REPO_ID, repo_type='model')
print(f'✅ Model at: https://huggingface.co/{HF_REPO_ID}')

---
## Appendix — Diagnostics

In [ ]:
# GPU memory at any point
!nvidia-smi --query-gpu=name,memory.used,memory.free --format=csv,noheader

In [ ]:
# Free GPU memory between runs
import torch, gc
try: del model
except: pass
gc.collect()
torch.cuda.empty_cache()
print(f'Available VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

In [ ]:
# Backup Qwen3 checkpoints to Drive
import shutil
from google.colab import drive
drive.mount('/content/drive')

BACKUP_DIR = '/content/drive/MyDrive/llm-ft/checkpoints_qwen3'
shutil.copytree('checkpoints', BACKUP_DIR, dirs_exist_ok=True)
print(f'✅ Checkpoints backed up to {BACKUP_DIR}')

In [ ]:
# Quick reference: what went wrong in Project 1 and what we fixed
lessons = [
    ('max_seq_length=2048', 'Truncated <answer> tags → format_think_pct dropped from 94% → 90.8%'),
    ('LR=5e-6',             'Loss still declining at step 450 → never converged in 3 epochs'),
    ('packing=True',        'Multi-example sequences caused boundary contamination with truncation'),
    ('raw text format',     'Manual eos_token workarounds → Unsloth placeholder conflict'),
    ('gradient_accum=16',   'Doubled effective batch without raising LR → even slower learning'),
]

print('=== Lessons from Project 1 ===')
for issue, effect in lessons:
    print(f'  ❌ {issue}')
    print(f'     → {effect}')
    print()

fixes = [
    ('max_seq_length=4096', '4-bit loading recovers the memory; full context for financial tables'),
    ('LR=2e-5',             'Fino1 model card value; loss should plateau before epoch 3'),
    ('packing=False',       'Each example is its own sequence; clean boundaries'),
    ('messages format',     'TRL + Qwen3 handle chat template automatically'),
    ('grad_accum=8',        'Effective batch=16 matches Fino1; more gradient updates per epoch'),
]

print('=== What Fino1 approach fixes ===')
for fix, reason in fixes:
    print(f'  ✅ {fix}')
    print(f'     → {reason}')
    print()